# **Análisis de Datos Saber 11 - Departamento de Caldas**

Este notebook implementa el proceso de selección, limpieza, alistamiento y análisis exploratorio de los datos de las pruebas Saber 11 para el departamento de Caldas, como parte del Proyecto 2 del curso *Analítica Computacional para la Toma de Decisiones*. El producto final está orientado al **Ministerio de Educación** como usuario final, y busca responder tres preguntas de negocio relacionadas con equidad socioeconómica, desempeño territorial y brechas de género.

## Tarea 2 - Selección, limpieza y alistamiento de datos

Los datos provienen del portal de [Datos Abiertos de Colombia](https://www.datos.gov.co/Educaci-n/Resultados-nicos-Saber-11/kgxf-xxbe), actualizados a abril de 2024. La selección, limpieza y alistamiento de datos ya fue realizada en el Proyecto 1.

## Tarea 3 - Exploración y análisis de datos

Con los datos limpios y correctamente tipados, se realiza un análisis de datos orientado a responder las tres preguntas de negocio que hemos desarrollado. Esto incluye estadísticas descriptivas, histogramas, diagramas de caja, diagramas de dispersión y mapas de calor sobre distintas variables relevantes para el desarrollo de los modelos de redes neuronales en tarea4/.

---

Daniel Benavides - 202220428 

Juanita Cortés - 202222129 

Andrés Felipe Herrera - 202220888

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from prettytable import PrettyTable

from matplotlib import font_manager
plt.rcParams['font.family'] = 'Arial'

alt.data_transformers.enable('vegafusion')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Carga de datos

In [ ]:
df = pd.read_csv('../data/caldas_data.csv')

print(f"Filas: {len(df)} | Columnas: {df.shape[1]}")
df.head(5)

## 1. Pregunta de negocio

### *¿Cuál es el puntaje global esperado para un estudiante de Caldas dado su perfil socioeconómico y el tipo de institución educativa?*

Esta pregunta se responde con un **modelo de regresión**, ya que la variable objetivo es continua:

$$
Y = punt\_global
$$

El objetivo es estimar el puntaje global esperado de un estudiante del departamento de Caldas a partir de variables asociadas a su contexto socioeconómico y a las características de la institución educativa. Esta predicción puede apoyar la identificación de perfiles con menor desempeño esperado y orientar estrategias de acompañamiento académico.

In [ ]:
# Se trabaja sobre una copia del dataframe original para no modificar las otras preguntas
df_p1 = df.copy()

# Variable objetivo de la pregunta 1
TARGET_P1 = 'punt_global'

# Eliminamos registros sin puntaje global porque no sirven para entrenar ni evaluar el modelo
df_p1 = df_p1.dropna(subset=[TARGET_P1])

print(f"Filas disponibles para la Pregunta 1: {df_p1.shape[0]}")
print(f"Columnas disponibles: {df_p1.shape[1]}")
df_p1[[TARGET_P1]].head()

In [ ]:
# --- Estrato: ordinal numérico 1-6 ---
# Se convierte el estrato a número porque tiene un orden natural.

estrato_map = {
    'Estrato 1': 1,
    'Estrato 2': 2,
    'Estrato 3': 3,
    'Estrato 4': 4,
    'Estrato 5': 5,
    'Estrato 6': 6
}

df_p1['estrato_num'] = df_p1['fami_estratovivienda'].map(estrato_map)

df_p1[['fami_estratovivienda', 'estrato_num']].head()

In [ ]:
# --- Educación de los padres: ordinal numérico ---
# Se asigna un valor numérico conservando el orden del nivel educativo.
# Las respuestas 'No sabe' y 'No aplica' quedan como NaN para no introducir ruido artificial.

edu_map = {
    'Ninguno': 0,
    'Primaria incompleta': 1,
    'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'Educación profesional completa': 8,
    'Postgrado': 9
}

df_p1['edu_madre_num'] = df_p1['fami_educacionmadre'].map(edu_map)
df_p1['edu_padre_num'] = df_p1['fami_educacionpadre'].map(edu_map)

df_p1[['fami_educacionmadre', 'edu_madre_num', 
       'fami_educacionpadre', 'edu_padre_num']].head()

In [ ]:
# --- Índice de activos del hogar ---
# Se construye un índice simple que resume disponibilidad de recursos en el hogar.
# Incluye computador, internet, automóvil y lavadora.

asset_cols = [
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tieneautomovil',
    'fami_tienelavadora'
]

df_p1['indice_activos'] = df_p1[asset_cols].sum(axis=1, skipna=True)

df_p1[asset_cols + ['indice_activos']].head()

In [ ]:
# --- Variables escolares ---
# Naturaleza del colegio: Público -> 0, Privado -> 1
# Área del colegio: RURAL -> 0, URBANO -> 1

df_p1['es_privado'] = df_p1['cole_naturaleza'].map({
    'Público': 0,
    'Privado': 1
})

df_p1['es_urbano'] = df_p1['cole_area_ubicacion'].map({
    'RURAL': 0,
    'URBANO': 1
})

df_p1[['cole_naturaleza', 'es_privado', 
       'cole_area_ubicacion', 'es_urbano']].head()

In [ ]:
# --- Género ---
# F -> 0, M -> 1

df_p1['genero_num'] = df_p1['estu_genero'].map({
    'F': 0,
    'M': 1
})

df_p1[['estu_genero', 'genero_num']].head()

crear df_model_p1

In [ ]:
# Variables seleccionadas para la Pregunta 1

FEATURES_P1 = [
    'estrato_num',
    'edu_madre_num',
    'edu_padre_num',
    'indice_activos',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tieneautomovil',
    'fami_tienelavadora',
    'es_privado',
    'es_urbano',
    'genero_num'
]

TARGET_P1 = 'punt_global'

df_model_p1 = df_p1[FEATURES_P1 + [TARGET_P1]].copy()

print(f"Shape del dataset exploratorio P1: {df_model_p1.shape}")
df_model_p1.head()

buscamos faltantes

In [ ]:
# Revisión de datos faltantes

missing_p1 = df_model_p1.isnull().mean().sort_values(ascending=False) * 100

missing_table_p1 = PrettyTable()
missing_table_p1.field_names = ["Variable", "% faltante"]

for col, pct in missing_p1.items():
    missing_table_p1.add_row([col, f"{pct:.2f}%"])

print(missing_table_p1)

### 1.3 Distribución del puntaje global

Se analiza la distribución de la variable objetivo `punt_global`, ya que esta será la variable a predecir en la etapa de modelamiento. Esta exploración permite identificar el rango general de puntajes, su concentración y posibles valores extremos.

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df_model_p1['punt_global'].dropna(), bins=30)
plt.xlabel('Puntaje global')
plt.ylabel('Frecuencia')
plt.title('Distribución del puntaje global en Caldas')
plt.show()

df_model_p1['punt_global'].describe()

### 1.4 Puntaje global promedio por estrato socioeconómico

Se compara el puntaje global promedio entre estratos para observar si existen diferencias en el desempeño esperado según el perfil socioeconómico del estudiante.

In [ ]:
puntaje_estrato = df_p1.groupby('estrato_num')['punt_global'].mean().sort_index()

plt.figure(figsize=(8,5))
puntaje_estrato.plot(kind='bar')
plt.xlabel('Estrato socioeconómico')
plt.ylabel('Puntaje global promedio')
plt.title('Puntaje global promedio por estrato')
plt.show()

puntaje_estrato

### 1.5 Puntaje global promedio por educación de los padres

Se analiza la relación entre el nivel educativo de los padres y el puntaje global promedio. Esta variable es relevante porque puede capturar diferencias en capital educativo del hogar.

In [ ]:
puntaje_madre = df_p1.groupby('edu_madre_num')['punt_global'].mean().sort_index()
puntaje_padre = df_p1.groupby('edu_padre_num')['punt_global'].mean().sort_index()

plt.figure(figsize=(8,5))
puntaje_madre.plot(kind='bar')
plt.xlabel('Nivel educativo de la madre')
plt.ylabel('Puntaje global promedio')
plt.title('Puntaje global promedio por educación de la madre')
plt.show()

plt.figure(figsize=(8,5))
puntaje_padre.plot(kind='bar')
plt.xlabel('Nivel educativo del padre')
plt.ylabel('Puntaje global promedio')
plt.title('Puntaje global promedio por educación del padre')
plt.show()

puntaje_madre, puntaje_padre

### 1.6 Puntaje global por tipo de colegio

Se analiza si existen diferencias en el puntaje global promedio según la naturaleza del colegio. Esta comparación permite observar posibles brechas entre instituciones públicas y privadas.

In [ ]:
puntaje_naturaleza = df_p1.groupby('cole_naturaleza')['punt_global'].mean().sort_values(ascending=False)

plt.figure(figsize=(7,5))
puntaje_naturaleza.plot(kind='bar')
plt.xlabel('Naturaleza del colegio')
plt.ylabel('Puntaje global promedio')
plt.title('Puntaje global promedio por naturaleza del colegio')
plt.xticks(rotation=0)
plt.show()

puntaje_naturaleza

### 1.7 Puntaje global por acceso a computador e internet

Se analiza el puntaje global promedio según el acceso del estudiante a computador e internet en el hogar. Estas variables son relevantes porque representan recursos que pueden facilitar el estudio, la búsqueda de información y la preparación académica.

In [ ]:
# Puntaje global promedio según acceso a computador

puntaje_computador = df_p1.groupby('fami_tienecomputador')['punt_global'].mean()

plt.figure(figsize=(7,5))
puntaje_computador.plot(kind='bar')
plt.xlabel('Tiene computador')
plt.ylabel('Puntaje global promedio')
plt.title('Puntaje global promedio según acceso a computador')
plt.xticks(rotation=0)
plt.show()

puntaje_computador

In [ ]:
# Puntaje global promedio según acceso a internet

puntaje_internet = df_p1.groupby('fami_tieneinternet')['punt_global'].mean()

plt.figure(figsize=(7,5))
puntaje_internet.plot(kind='bar')
plt.xlabel('Tiene internet')
plt.ylabel('Puntaje global promedio')
plt.title('Puntaje global promedio según acceso a internet')
plt.xticks(rotation=0)
plt.show()

puntaje_internet

### 1.8 Correlación entre variables seleccionadas y puntaje global

Finalmente, se calcula la correlación entre las variables numéricas seleccionadas y el puntaje global. Este análisis ayuda a identificar qué variables presentan una mayor asociación lineal con la variable objetivo.

In [ ]:
FEATURES_P1 = [
    'estrato_num',
    'edu_madre_num',
    'edu_padre_num',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'es_privado',
    'es_urbano'
]

TARGET_P1 = 'punt_global'

df_model_p1 = df_p1[FEATURES_P1 + [TARGET_P1]].copy()

In [ ]:
# Renombrar variables para que la matriz sea más legible

rename_vars_p1 = {
    'estrato_num': 'Estrato',
    'edu_madre_num': 'Educación madre',
    'edu_padre_num': 'Educación padre',
    'fami_tienecomputador': 'Tiene computador',
    'fami_tieneinternet': 'Tiene internet',
    'es_privado': 'Colegio privado',
    'es_urbano': 'Colegio urbano',
    'punt_global': 'Puntaje global'
}

corr_matrix_p1 = df_model_p1.rename(columns=rename_vars_p1).corr(numeric_only=True)

plt.figure(figsize=(9,6))
sns.heatmap(
    corr_matrix_p1,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlación'}
)

plt.title('Matriz de correlación - Pregunta 1')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Pregunta de negocio

### *¿Puede identificarse si un estudiante está en riesgo de obtener un puntaje global por debajo del umbral de bajo desempeño, según sus características socioeconómicas y escolares?*

Esta pregunta se responde con un **modelo de clasificación binaria**. La variable objetivo es:

$$\text{bajo\_rendimiento} = \begin{cases} 1 & \text{si } \texttt{punt\_global} < 250 \\ 0 & \text{en otro caso} \end{cases}$$

El umbral de 220 puntos es consistente con el análisis del Proyecto 1, donde se identificó ese valor como frontera entre municipios de rendimiento crítico y rendimiento medio-alto.

### 2.1 Feature Engineering

In [ ]:
UMBRAL_BAJO = 220

# --- Target variable ---
# Se define la variable objetivo como clasificación binaria
# Se eliminan filas sin punt_global ya que es indispensable para el target

df = df.dropna(subset=['punt_global'])
df['bajo_rendimiento'] = (df['punt_global'] < UMBRAL_BAJO).astype(int)

print(f"Distribución del target:\n{df['bajo_rendimiento'].value_counts()}")
print(f"\n% bajo rendimiento: {df['bajo_rendimiento'].mean():.1%}")

In [ ]:
# --- Estrato: ordinal numérico 1-6 ---
# El P1 mostró que el estrato tiene efecto progresivo y monotónico sobre el puntaje,
# por lo que es apropiado tratarlo como ordinal numérico en lugar de dummies.
estrato_map = {
    'Estrato 1': 1, 'Estrato 2': 2, 'Estrato 3': 3,
    'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6
}
df['estrato_num'] = df['fami_estratovivienda'].map(estrato_map)

# --- Educación de los padres: ordinal numérico ---
# El P1 mostró un gradiente continuo: a mayor educación, mayor puntaje.
# Asignamos un entero que preserva el orden natural del nivel educativo.
# 'No sabe' / 'No aplica' → NaN para no introducir ruido.
edu_map = {
    'Ninguno': 0,
    'Primaria incompleta': 1,
    'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3,
    'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5,
    'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7,
    'Educación profesional completa': 8,
    'Postgrado': 9,
}

df['edu_madre_num'] = df['fami_educacionmadre'].map(edu_map)   # NaN si 'No sabe'/'No aplica'
df['edu_padre_num'] = df['fami_educacionpadre'].map(edu_map)

# --- Índice de activos del hogar (0-4) ---
# Creado en P1. Si no existe, se recalcula.
asset_cols = ['fami_tienecomputador', 'fami_tieneinternet',
              'fami_tieneautomovil', 'fami_tienelavadora']
if 'indice_activos' not in df.columns:
    df['indice_activos'] = df[asset_cols].sum(axis=1, skipna=True)

# --- Variables escolares binarias ---
# cole_naturaleza: Público → 0, Privado → 1
df['es_privado'] = (df['cole_naturaleza'] == 'Privado').astype(float)
df.loc[df['cole_naturaleza'].isna(), 'es_privado'] = np.nan

# cole_area_ubicacion: RURAL → 0, URBANO → 1
df['es_urbano'] = (df['cole_area_ubicacion'] == 'URBANO').astype(float)
df.loc[df['cole_area_ubicacion'].isna(), 'es_urbano'] = np.nan

# --- Género: M → 1, F → 0 ---
df['genero_num'] = df['estu_genero'].map({'M': 1, 'F': 0})

In [ ]:
# Definir el conjunto de features que usará el modelo

FEATURES = [
    'estrato_num',
    'edu_madre_num',
    'edu_padre_num',
    'indice_activos',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tieneautomovil',
    'fami_tienelavadora',
    'es_privado',
    'es_urbano',
    'genero_num',
]

TARGET = 'bajo_rendimiento'

df_model = df[FEATURES + [TARGET]].copy()
print(f"Shape del dataset de modelamiento: {df_model.shape}")
df_model.head(3)

### 2.2 Datos faltantes

In [ ]:
missing = df_model.isnull().mean().sort_values(ascending=False) * 100
missing_table = PrettyTable()
missing_table.field_names = ["Feature", "% Faltante"]
for col, pct in missing.items():
    missing_table.add_row([col, f"{pct:.2f}%"])
print(missing_table)

### 2.3 Analysis

In [ ]:
# Estrato vs bajo_rendimiento

estrato_risk = (
    df_model.dropna(subset=['estrato_num'])
    .groupby('estrato_num')[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'tasa_riesgo'})
)

estrato_risk['estrato_num'] = estrato_risk['estrato_num'].astype(str)

chart_estrato = alt.Chart(estrato_risk).mark_bar().encode(
    x=alt.X('estrato_num:O', title='Estrato', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
    color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
).properties(
    width=350, height=280,
    title='Tasa de bajo rendimiento por estrato socioeconómico'
)

chart_estrato

In [ ]:
# Índice de activos vs tasa de riesgo.
# El P1 confirmó que a mayor índice de activos, mayor puntaje.

activos_risk = (
    df_model.dropna(subset=['indice_activos'])
    .groupby('indice_activos')[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'tasa_riesgo'})
)
activos_risk['indice_activos'] = activos_risk['indice_activos'].astype(str)

chart_activos = alt.Chart(activos_risk).mark_bar().encode(
    x=alt.X('indice_activos:O', title='Índice de activos del hogar (0 = ninguno, 4 = todos)',
            axis=alt.Axis(labelAngle=0)),
    y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
    color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
).properties(
    width=350, height=280,
    title='Tasa de bajo rendimiento por índice de activos del hogar'
)

chart_activos

In [ ]:
# Educación de la madre vs tasa de riesgo

edu_risk = (
    df_model.dropna(subset=['edu_madre_num'])
    .groupby('edu_madre_num')[TARGET]
    .mean()
    .reset_index()
    .rename(columns={TARGET: 'tasa_riesgo'})
)

edu_labels = {
    0: 'Ninguno', 1: 'Prim. inc.', 2: 'Prim. comp.',
    3: 'Sec. inc.', 4: 'Sec. comp.', 5: 'Téc. inc.',
    6: 'Téc. comp.', 7: 'Prof. inc.', 8: 'Prof. comp.', 9: 'Postgrado'
}
edu_risk['nivel'] = edu_risk['edu_madre_num'].map(edu_labels)

chart_edu = alt.Chart(edu_risk).mark_bar().encode(
    x=alt.X('edu_madre_num:O', title='Nivel educativo de la madre',
            axis=alt.Axis(labelExpr="{'0':'Ninguno','1':'Prim.inc','2':'Prim.comp','3':'Sec.inc','4':'Sec.comp','5':'Téc.inc','6':'Téc.comp','7':'Prof.inc','8':'Prof.comp','9':'Postgrado'}[datum.label]",
                          labelAngle=-35)),
    y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
    tooltip=['nivel:N', alt.Tooltip('tasa_riesgo:Q', format='.1%')],
    color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
).properties(
    width=400, height=280,
    title='Tasa de bajo rendimiento por nivel educativo de la madre'
)

chart_edu

In [ ]:
# Naturaleza del colegio y zona vs tasa de riesgo.
# El P1 mostró que privado > público en puntaje, y que la brecha rural-urbana no es uniforme

def tasa_por_binaria(col, label_0, label_1):
    tmp = (
        df_model.dropna(subset=[col])
        .groupby(col)[TARGET].mean()
        .reset_index()
        .rename(columns={col: 'valor', TARGET: 'tasa_riesgo'})
    )
    tmp['etiqueta'] = tmp['valor'].map({0.0: label_0, 1.0: label_1})
    return tmp

df_priv = tasa_por_binaria('es_privado', 'Público', 'Privado')
df_urb  = tasa_por_binaria('es_urbano', 'Rural', 'Urbano')

def bar_binaria(data, titulo):
    return alt.Chart(data).mark_bar().encode(
        x=alt.X('etiqueta:N', title='', axis=alt.Axis(labelAngle=0)),
        y=alt.Y('tasa_riesgo:Q', title='Tasa de bajo rendimiento', axis=alt.Axis(format='%')),
        color=alt.Color('tasa_riesgo:Q', scale=alt.Scale(scheme='reds'), legend=None)
    ).properties(width=200, height=260, title=titulo)

(bar_binaria(df_priv, 'Público vs Privado') | bar_binaria(df_urb, 'Rural vs Urbano'))

### 2.4 Correlación de features con el target

In [ ]:
# Mapa de calor de correlaciones entre todos los features

corr_matrix = df_model.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax
)
ax.set_title('Matriz de correlación - features del modelo Q2', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Pregunta de negocio

### *¿Puede predecirse el nivel de desempeño en inglés de un estudiante (A−, A1, A2, B1, B+) a partir de su perfil académico, socioeconómico y de género?*

### 3.1 Feature Enginnering

In [ ]:
# Variable objetivo: nivel de desempeño en inglés
TARGET_P3 = 'desemp_ingles'

# Eliminar filas sin target
df_p3 = df.dropna(subset=[TARGET_P3]).copy()

# Mantener únicamente las categorías esperadas
niveles_ingles = ['A-', 'A1', 'A2', 'B1', 'B+']
df_p3 = df_p3[df_p3[TARGET_P3].isin(niveles_ingles)]

print("Distribución del target:")
print(df_p3[TARGET_P3].value_counts())

print("\nDistribución porcentual:")
print((df_p3[TARGET_P3].value_counts(normalize=True) * 100).round(2))

### 3.2 Variables Relevantes

In [ ]:
# Variables académicas
academic_features = [
    'punt_matematicas',
    'punt_lectura_critica',
    'punt_c_naturales',
    'punt_sociales_ciudadanas']

# Variables socioeconómicas
socioeconomic_features = [
    'estrato_num',
    'edu_madre_num',
    'edu_padre_num',
    'indice_activos',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tieneautomovil',
    'fami_tienelavadora']

# Variable de género
gender_features = [
    'genero_num'
]

# Variables escolares opcionales
school_features = [
    'es_privado',
    'es_urbano',
    'cole_bilingue']

FEATURES_P3 = academic_features + socioeconomic_features + gender_features + school_features

df_model_p3 = df_p3[FEATURES_P3 + [TARGET_P3]].copy()

print(f"Shape del dataset de modelamiento P3: {df_model_p3.shape}")
df_model_p3.head()

### 3.3 Datos Faltantes

In [ ]:
missing_p3 = df_model_p3.isnull().mean().sort_values(ascending=False) * 100

missing_table_p3 = PrettyTable()
missing_table_p3.field_names = ["Feature", "% Faltante"]

for col, pct in missing_p3.items():
    missing_table_p3.add_row([col, f"{pct:.2f}%"])

print(missing_table_p3)

### 3.4 Analysis

In [ ]:
target_dist = (
    df_model_p3[TARGET_P3]
    .value_counts()
    .reindex(niveles_ingles)
    .reset_index()
)

target_dist.columns = ['nivel_ingles', 'conteo']
target_dist['porcentaje'] = target_dist['conteo'] / target_dist['conteo'].sum()

chart_target = alt.Chart(target_dist).mark_bar().encode(
    x=alt.X('nivel_ingles:O', title='Nivel de desempeño en inglés', sort=niveles_ingles),
    y=alt.Y('conteo:Q', title='Número de estudiantes'),
    tooltip=[
        alt.Tooltip('nivel_ingles:N', title='Nivel'),
        alt.Tooltip('conteo:Q', title='Estudiantes'),
        alt.Tooltip('porcentaje:Q', title='Porcentaje', format='.1%')
    ]
).properties(
    width=450,
    height=300,
    title='Distribución del nivel de desempeño en inglés'
)

chart_target

In [ ]:
academic_summary = (
    df_p3
    .groupby(TARGET_P3)[academic_features + ['punt_global']]
    .mean()
    .reindex(niveles_ingles)
    .reset_index()
)

academic_summary

In [ ]:
estrato_ingles = (
    df_p3
    .dropna(subset=['fami_estratovivienda'])
    .groupby(['fami_estratovivienda', TARGET_P3])
    .size()
    .reset_index(name='conteo')
)

estrato_ingles['porcentaje'] = (
    estrato_ingles
    .groupby('fami_estratovivienda')['conteo']
    .transform(lambda x: x / x.sum())
)

chart_estrato_ingles = alt.Chart(estrato_ingles).mark_bar().encode(
    x=alt.X('fami_estratovivienda:O', title='Estrato socioeconómico'),
    y=alt.Y('porcentaje:Q', title='Porcentaje', axis=alt.Axis(format='%')),
    color=alt.Color(f'{TARGET_P3}:N', title='Nivel inglés', sort=niveles_ingles),
    tooltip=[
        alt.Tooltip('fami_estratovivienda:N', title='Estrato'),
        alt.Tooltip(f'{TARGET_P3}:N', title='Nivel inglés'),
        alt.Tooltip('porcentaje:Q', title='Porcentaje', format='.1%'),
        alt.Tooltip('conteo:Q', title='Estudiantes')
    ]
).properties(
    width=600,
    height=350,
    title='Distribución del nivel de inglés por estrato socioeconómico'
)

chart_estrato_ingles

In [ ]:
genero_ingles = (
    df_p3
    .dropna(subset=['estu_genero'])
    .groupby(['estu_genero', TARGET_P3])
    .size()
    .reset_index(name='conteo')
)

genero_ingles['porcentaje'] = (
    genero_ingles
    .groupby('estu_genero')['conteo']
    .transform(lambda x: x / x.sum())
)

chart_genero_ingles = alt.Chart(genero_ingles).mark_bar().encode(
    x=alt.X('estu_genero:N', title='Género'),
    y=alt.Y('porcentaje:Q', title='Porcentaje', axis=alt.Axis(format='%')),
    color=alt.Color(f'{TARGET_P3}:N', title='Nivel inglés', sort=niveles_ingles),
    tooltip=[
        alt.Tooltip('estu_genero:N', title='Género'),
        alt.Tooltip(f'{TARGET_P3}:N', title='Nivel inglés'),
        alt.Tooltip('porcentaje:Q', title='Porcentaje', format='.1%'),
        alt.Tooltip('conteo:Q', title='Estudiantes')
    ]
).properties(
    width=350,
    height=300,
    title='Distribución del nivel de inglés por género'
)

chart_genero_ingles

In [ ]:
# Codificación solo para análisis exploratorio
ingles_ord_map = {
    'A-': 0,
    'A1': 1,
    'A2': 2,
    'B1': 3,
    'B+': 4
}

df_model_p3['ingles_ord'] = df_model_p3[TARGET_P3].map(ingles_ord_map)

corr_p3 = (
    df_model_p3
    .drop(columns=[TARGET_P3])
    .corr(numeric_only=True)['ingles_ord']
    .sort_values(ascending=False)
    .drop('ingles_ord')
)

corr_p3

In [ ]:
corr_p3_df = corr_p3.reset_index()
corr_p3_df.columns = ['feature', 'correlacion']

chart_corr_p3 = alt.Chart(corr_p3_df).mark_bar().encode(
    x=alt.X('correlacion:Q', title='Correlación con nivel de inglés codificado'),
    y=alt.Y('feature:N', title='Variable', sort='-x'),
    tooltip=[
        alt.Tooltip('feature:N', title='Variable'),
        alt.Tooltip('correlacion:Q', title='Correlación', format='.2f')
    ]
).properties(
    width=550,
    height=400,
    title='Relación de variables numéricas con el nivel de inglés'
)

chart_corr_p3

## Dataset consolidado para modelamiento

Se genera un único CSV con todos los features utilizados en las tres preguntas de negocio, junto con las tres variables objetivo. Este archivo es el punto de entrada para tarea4/.

In [ ]:
UMBRAL_BAJO = 230

df_master = df.copy()

# Features compartidos (Q1, Q2, Q3)
estrato_map = {
    'Estrato 1': 1, 'Estrato 2': 2, 'Estrato 3': 3,
    'Estrato 4': 4, 'Estrato 5': 5, 'Estrato 6': 6
}
edu_map = {
    'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
    'Secundaria (Bachillerato) incompleta': 3, 'Secundaria (Bachillerato) completa': 4,
    'Técnica o tecnológica incompleta': 5, 'Técnica o tecnológica completa': 6,
    'Educación profesional incompleta': 7, 'Educación profesional completa': 8,
    'Postgrado': 9,
}

df_master['estrato_num']   = df_master['fami_estratovivienda'].map(estrato_map)
df_master['edu_madre_num'] = df_master['fami_educacionmadre'].map(edu_map)
df_master['edu_padre_num'] = df_master['fami_educacionpadre'].map(edu_map)

asset_cols = ['fami_tienecomputador', 'fami_tieneinternet',
              'fami_tieneautomovil', 'fami_tienelavadora']
df_master['indice_activos'] = df_master[asset_cols].sum(axis=1, skipna=True)

df_master['es_privado'] = df_master['cole_naturaleza'].map({'Público': 0, 'Privado': 1})
df_master['es_urbano']  = df_master['cole_area_ubicacion'].map({'RURAL': 0, 'URBANO': 1})
df_master['genero_num'] = df_master['estu_genero'].map({'F': 0, 'M': 1})

# Targets
df_master['bajo_rendimiento'] = (df_master['punt_global'] < UMBRAL_BAJO).astype('Int64')

COLS_FINALES = [
    # Contexto
    'periodo', 'cole_mcpio_ubicacion',
    # Features socioeconómicos
    'estrato_num', 'edu_madre_num', 'edu_padre_num', 'indice_activos',
    'fami_tienecomputador', 'fami_tieneinternet',
    'fami_tieneautomovil', 'fami_tienelavadora',
    # Features escolares
    'es_privado', 'es_urbano', 'cole_bilingue',
    # Feature género
    'genero_num',
    # Scores académicos (features para Q3)
    'punt_matematicas', 'punt_lectura_critica',
    'punt_c_naturales', 'punt_sociales_ciudadanas', 'punt_ingles',
    # Targets
    'punt_global',        # Q1
    'bajo_rendimiento',   # Q2
    'desemp_ingles',      # Q3
]

df_master = df_master[COLS_FINALES]

print(f"Shape: {df_master.shape}")
print(f"Faltantes por columna:\n{df_master.isnull().sum()[df_master.isnull().sum() > 0]}")

df_master.to_csv('../data/saber11_features.csv', index=False)
print("\nGuardado en ../data/saber11_features.csv")